# SPS MBB Dipole -- Comprehensive Analysis (26 GeV)

**Measurement session:** `MBB/2026-02-06_supercycle/03_26_extended/20260206_151808_SPS_MBB`
**Segments:** NCS
**Magnet:** MBB (normal dipole, m=1)
### Part I: Setup & Data Quality
| # | Section |
|---|---------|
| 1 | Configuration & Imports |
| 2 | Kn Calibration |
| 3 | Data Loading & Channel Detection |
| 4 | Raw Signals Overview |
| 5 | cel/fed Safety Diagnostic |
| 6 | Plateau Detection & Turn Classification |
| 7 | FDI Stuck-Channel Diagnostic |

### Part II: Pipeline Processing
| # | Section |
|---|---------|
| 8 | Process Plateau Turns |
| 9 | All-Turn Harmonics vs Time |
| 10 | FFMM Golden Standard Validation |

### Part III: Harmonic Analysis
| # | Section |
|---|---------|
| 11 | Main Field (B1) |
| 12 | b2 (Quadrupole) |
| 13 | b3 (Sextupole) |
| 14 | Higher Harmonics Overview |
| 15 | Multipole Spectrum |

### Part IV: Transfer Function & Inductance
| # | Section |
|---|---------|
| 16 | Transfer Function B1/I |
| 17 | Apparent vs Differential Inductance |

### Part V: Eddy Current & Settling
| # | Section |
|---|---------|
| 18 | Eddy Current Settling Analysis |
| 19 | Exponential Fits |

### Part VI: Summary
| # | Section |
|---|---------|
| 20 | Outlier-Cleaned Fits |

### Part VII: Eddy Current & Settling
| # | Section |
|---|---------|
| 21 | Settling Bias Analysis |
| 22 | N_LAST Sensitivity Study |

### Part VIII: Summary
| # | Section |
|---|---------|
| 23 | Comprehensive Statistics Table |
| 24 | Analysis Choices Summary |
| 25 | CSV Export |

---
## 1. Configuration & Imports

In [ ]:
# === CONFIGURATION ===
SEGMENT_CONFIGS = [
    {"name": "NCS", "kn_path": "MBB/2025-12-12/CRMMMMH_AV-00000001/Kn_values_Seg_Main_A_AC.txt", "merge_mode": "abs_upto_m_cmp_above", "is_fringe": False},
]
SEGMENTS = [s["name"] for s in SEGMENT_CONFIGS]
SEG_DISPLAY = {s["name"]: (s["name"] + " [fringe]" if s["is_fringe"] else s["name"] + " [main body]") for s in SEGMENT_CONFIGS}

SESSION = "MBB/2026-02-06_supercycle/03_26_extended/20260206_151808_SPS_MBB"
MEAS_SUBDIR = "20260206_151827_MBB"
KN_PATHS = {"NCS": "MBB/2025-12-12/CRMMMMH_AV-00000001/Kn_values_Seg_Main_A_AC.txt"}

MAGNET_ORDER = 1
R_REF = 0.02
L_COIL = 0.47
SAMPLES_PER_TURN = 1024

OPTIONS = ('dri', 'rot', 'cel', 'fed')
MIN_B1_T = 0.0001
PLATEAU_I_RANGE_MAX = 3.0
N_BLOCKS = 10

N_LAST_TURNS_INJ = 18
N_LAST_TURNS_HIGH = None

N_SIGMA_CLIP = 5.0
MIN_INJECTION_TURNS = 5

print("SPS MBB Dipole -- 26 GeV MD1 Extended NCS")
print("=" * 60)
print(f"  Segments      : {SEGMENTS}")
print(f"  Magnet order  : {MAGNET_ORDER}")
print(f"  R_ref         : {R_REF} m")
print(f"  Samples/turn  : {SAMPLES_PER_TURN}")
print(f"  Options       : {OPTIONS}")

In [ ]:
import sys
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.optimize import curve_fit

%matplotlib widget
plt.rcParams.update({
    "figure.figsize": (14, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "figure.dpi": 100,
})

REPO_ROOT = Path(".").resolve()
while REPO_ROOT != REPO_ROOT.parent:
    if (REPO_ROOT / "pyproject.toml").exists() or (REPO_ROOT / ".git").exists():
        break
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from rotating_coil_analyzer.analysis.kn_pipeline import load_segment_kn_txt
from rotating_coil_analyzer.analysis.utility_functions import (
    process_kn_pipeline,
    build_harmonic_rows,
    diagnose_cel_fed,
    mad_sigma_clip,
    eddy_model,
    fit_eddy_per_run,
    plateau_summary,
    compute_block_averaged_range,
    detect_plateau_turns,
    classify_current,
    find_contiguous_groups,
    diagnose_fdi_transitions,
)
from rotating_coil_analyzer.ingest.channel_detect import robust_range

SESSION_DIR = REPO_ROOT / "measurements" / SESSION
RUN_DIR = SESSION_DIR / MEAS_SUBDIR

KN = {}
for _seg_name, _kn_rel in KN_PATHS.items():
    _kp = REPO_ROOT / "measurements" / _kn_rel
    assert _kp.exists(), f"Kn file not found: {_kp}"
    KN[_seg_name] = load_segment_kn_txt(str(_kp))

print(f"Repo root : {REPO_ROOT}")
print(f"Kn loaded : {list(KN.keys())}")
print("Imports ready.")

---
## 2. Kn Calibration

In [ ]:
for seg_name, kn_seg in KN.items():
    H = len(kn_seg.orders)
    print(f"\n{seg_name}: {H} harmonics")
    print(f"  Orders: {list(kn_seg.orders)}")
    kn_abs_n1 = abs(kn_seg.kn_abs[0])
    kn_cmp_n1 = abs(kn_seg.kn_cmp[0])
    ratio = kn_abs_n1 / max(kn_cmp_n1, 1e-30)
    print(f"  |Kn_abs(n=1)| = {kn_abs_n1:.6e}")
    print(f"  |Kn_cmp(n=1)| = {kn_cmp_n1:.6e}")
    print(f"  Abs/Cmp ratio (n=1): {ratio:.0f}x")

# Use first segment's Kn for harmonic count
_first_seg = SEGMENTS[0]
H = len(KN[_first_seg].orders)
Ns = SAMPLES_PER_TURN
m = MAGNET_ORDER
print(f"\nH={H}, Ns={Ns}, m={m}")

---
## 3. Data Loading & Channel Detection

Load raw measurement data for all segments.

In [ ]:
FILE_PAT = re.compile(
    r"Run_(\d+)_I_([\d.]+)A_(N?CS)_raw_measurement_data\.txt$"
)

data = {}
for seg in SEGMENTS:
    seg_files = [
        f for f in sorted(RUN_DIR.iterdir())
        if FILE_PAT.search(f.name) and FILE_PAT.search(f.name).group(3) == seg
    ]
    assert seg_files, f"No {seg} raw files in {RUN_DIR}"
    raw_file = seg_files[0]

    raw = np.loadtxt(raw_file)
    n_turns = raw.shape[0] // Ns
    n_keep = n_turns * Ns
    ncols = raw.shape[1]

    t_all = raw[:n_keep, 0].reshape(n_turns, Ns)
    flux_col1 = raw[:n_keep, 1].reshape(n_turns, Ns)
    flux_col2 = raw[:n_keep, 2].reshape(n_turns, Ns)
    I_all = raw[:n_keep, 3].reshape(n_turns, Ns)

    # Auto-detect channel swap
    I_mean_quick = I_all.mean(axis=1)
    best_turn = np.argmax(np.abs(I_mean_quick))
    r1 = robust_range(raw[best_turn * Ns:(best_turn + 1) * Ns, 1])
    r2 = robust_range(raw[best_turn * Ns:(best_turn + 1) * Ns, 2])
    swap = r2 > r1

    if swap:
        flux_abs_all, flux_cmp_all = flux_col2, flux_col1
    else:
        flux_abs_all, flux_cmp_all = flux_col1, flux_col2

    data[seg] = {
        "raw_file": raw_file, "n_turns": n_turns,
        "t_all": t_all, "flux_abs": flux_abs_all, "flux_cmp": flux_cmp_all,
        "I_all": I_all, "swap": swap, "r1": r1, "r2": r2,
    }
    _scfg = next(sc for sc in SEGMENT_CONFIGS if sc["name"] == seg)
    fringe_tag = " [FRINGE FIELD]" if _scfg["is_fringe"] else ""
    print(f"\n{seg}{fringe_tag}: {raw_file.name}")
    print(f"  Shape: {raw.shape} -> {n_turns} turns, {ncols} columns")
    print(f"  Time span: {raw[-1,0] - raw[0,0]:.1f} s ({(raw[-1,0] - raw[0,0])/60:.1f} min)")
    print(f"  Flux swap: {swap}  (abs range={max(r1,r2):.4e}, cmp range={min(r1,r2):.4e})")

---
## 4. Raw Signals Overview

In [ ]:
fig, axes = plt.subplots(len(SEGMENTS), 3, figsize=(18, 5 * len(SEGMENTS)), sharex="col")
if len(SEGMENTS) == 1:
    axes = axes[np.newaxis, :]

for i, seg in enumerate(SEGMENTS):
    d = data[seg]
    n_keep = d["n_turns"] * Ns
    x = np.arange(n_keep)
    axes[i, 0].plot(x, d["flux_abs"].ravel(), linewidth=0.2, color="steelblue")
    axes[i, 0].set_ylabel(f"Flux abs ({seg})")
    axes[i, 0].set_title(f"Absolute flux -- {seg}")
    axes[i, 1].plot(x, d["flux_cmp"].ravel(), linewidth=0.2, color="teal")
    axes[i, 1].set_ylabel(f"Flux cmp ({seg})")
    axes[i, 1].set_title(f"Compensated flux -- {seg}")
    axes[i, 2].plot(x, d["I_all"].ravel(), linewidth=0.2, color="tab:orange")
    axes[i, 2].set_ylabel(f"Current ({seg})")
    axes[i, 2].set_title(f"Current -- {seg}")

axes[-1, 0].set_xlabel("Sample index")
axes[-1, 1].set_xlabel("Sample index")
axes[-1, 2].set_xlabel("Sample index")
fig.suptitle("Raw signals", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 5. cel/fed Safety Diagnostic

Run `diagnose_cel_fed()` on NCS high-current turns.

In [ ]:
# Diagnostic on main segment (not fringe)
_main_seg = "NCS"
d = data[_main_seg]
I_mean = d["I_all"].mean(axis=1)
hi_mask = np.abs(I_mean) > np.percentile(np.abs(I_mean), 90)
if hi_mask.sum() < 5:
    hi_mask = np.abs(I_mean) > np.median(np.abs(I_mean))

n_diag = min(100, int(hi_mask.sum()))
if n_diag == 0:
    print(f"No high-I turns in {_main_seg} for cel/fed diagnostic -- skipping")
else:
    hi_idx = np.where(hi_mask)[0][:n_diag]

    diag = diagnose_cel_fed(
        d["flux_abs"][hi_idx], d["flux_cmp"][hi_idx],
        d["t_all"][hi_idx], d["I_all"][hi_idx],
        kn=KN[_main_seg], r_ref=R_REF, magnet_order=MAGNET_ORDER,
    )
    print(f"cel/fed diagnostic ({n_diag} {_main_seg} high-I turns):")
    print(f"  Recommendation: {diag.recommendation}")
    print(f"  {diag.reason}")
    Bd = np.max(np.abs(diag.B_main_with_fed - diag.B_main_without_fed))
    print(f"  B_main max |diff|: {Bd:.4e} T")

    if diag.recommendation == "UNSAFE":
        OPTIONS = tuple(o for o in OPTIONS if o not in ("cel", "fed"))
        print(f"  -> cel/fed disabled, OPTIONS = {OPTIONS}")
    else:
        print(f"  -> cel/fed safe, keeping OPTIONS = {OPTIONS}")

---
## 6. Plateau Detection & Turn Classification

In [ ]:
label_colors = {"injection": "tab:green", "flat-mid": "tab:purple", "flat-high": "tab:blue"}

for seg in SEGMENTS:
    d = data[seg]
    I_mean = d["I_all"].mean(axis=1)
    t_mean = d["t_all"].mean(axis=1)
    I_range, I_blocks = compute_block_averaged_range(d["I_all"], Ns, N_BLOCKS)

    plateau_info = detect_plateau_turns(I_blocks, I_mean, I_range, PLATEAU_I_RANGE_MAX)
    is_plateau = plateau_info["is_plateau"]

    turn_label = np.array(["ramp"] * d["n_turns"], dtype=object)
    for j in range(d["n_turns"]):
        if is_plateau[j]:
            turn_label[j] = classify_current(I_mean[j])

    inj_groups = find_contiguous_groups(turn_label == "injection", min_length=2)
    fh_groups = find_contiguous_groups(turn_label == "flat-high", min_length=2)

    d.update({
        "I_mean": I_mean, "t_mean": t_mean, "I_range": I_range,
        "is_plateau": is_plateau, "turn_label": turn_label,
        "inj_groups": inj_groups, "fh_groups": fh_groups,
    })

    _scfg = next(sc for sc in SEGMENT_CONFIGS if sc["name"] == seg)
    fringe = " [FRINGE]" if _scfg["is_fringe"] else ""
    print(f"\n{seg}{fringe}: {is_plateau.sum()} plateau, "
          f"{len(inj_groups)} inj groups, {len(fh_groups)} flat-high groups")
    for lab in ["injection", "flat-mid", "flat-high"]:
        mask = turn_label == lab
        if mask.sum() > 0:
            print(f"  {lab:12s}: {mask.sum():4d} turns, I = {I_mean[mask].mean():.1f} +/- {I_mean[mask].std():.1f} A")
    print(f"  {'ramp':12s}: {(turn_label == 'ramp').sum():4d} turns")

# Plot
fig, axes = plt.subplots(1, len(SEGMENTS), figsize=(8 * len(SEGMENTS), 5), sharey=True)
if len(SEGMENTS) == 1:
    axes = [axes]
for ax, seg in zip(axes, SEGMENTS):
    d = data[seg]
    ax.plot(d["t_mean"], d["I_mean"], ".-", markersize=1, linewidth=0.3, color="lightgrey", zorder=0)
    for lab, col in label_colors.items():
        mask = d["turn_label"] == lab
        idx = np.where(mask)[0]
        if len(idx) > 0:
            ax.scatter(d["t_mean"][idx], d["I_mean"][idx], s=6, color=col, zorder=2, label=lab)
    ax.set_xlabel("Time (s)"); ax.set_ylabel("I (A)")
    ax.set_title(f"Plateau Detection -- {seg}"); ax.legend(fontsize=9)
fig.suptitle("Current Profile & Plateau Detection", fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

---
## 7. FDI Stuck-Channel Diagnostic

Check whether the FDI responds to current changes between plateau groups.

In [ ]:
for seg in SEGMENTS:
    d = data[seg]
    # Build run_info from contiguous plateau groups
    all_groups = []
    for lab_name in ["injection", "flat-mid", "flat-high"]:
        groups = find_contiguous_groups(d["turn_label"] == lab_name, min_length=2)
        for gs, ge in groups:
            all_groups.append({"start": gs, "end": ge,
                               "I_nom": float(d["I_mean"][gs:ge+1].mean())})
    all_groups.sort(key=lambda x: x["start"])
    for i, g in enumerate(all_groups):
        g["run_id"] = i

    if len(all_groups) < 2:
        print(f"{seg}: fewer than 2 plateau groups, skipping FDI check")
        continue

    flux_turns = d["flux_abs"].mean(axis=1)
    checks = diagnose_fdi_transitions(
        flux_turns, d["I_mean"], all_groups,
        stuck_threshold=0.3, partial_threshold=0.7, min_delta_I=5.0,
    )
    n_ok = sum(1 for c in checks if c.severity == "OK")
    n_stuck = sum(1 for c in checks if c.severity == "STUCK")
    print(f"\n{seg}: {len(checks)} transitions, OK={n_ok}, STUCK={n_stuck}")
    for c in checks:
        if c.severity != "OK":
            print(f"  ! Run {c.run_before}->{c.run_after}: {c.severity} -- {c.reason}")
    if n_stuck > 0:
        print(f"  WARNING: {n_stuck} stuck transitions!")
    else:
        print(f"  All transitions OK.")

---
## 8. Process Plateau Turns

Re-process plateau turns with the full pipeline. Group by supercycle/run, apply settling window and MAD sigma-clip.

In [ ]:
ANALYSIS_LABELS = {"injection", "flat-mid", "flat-high"}

results = {}

for seg in SEGMENTS:
    d = data[seg]
    turn_label = d["turn_label"]
    kn_seg = KN[seg]

    is_analysis = np.array([l in ANALYSIS_LABELS for l in turn_label])
    plateau_indices = np.where(is_analysis)[0]
    print(f"\n{seg}: processing {len(plateau_indices)} plateau turns (OPTIONS={OPTIONS})")

    result, C_merged, C_units, ok_main = process_kn_pipeline(
        flux_abs_turns=d["flux_abs"][plateau_indices],
        flux_cmp_turns=d["flux_cmp"][plateau_indices],
        t_turns=d["t_all"][plateau_indices],
        I_turns=d["I_all"][plateau_indices],
        kn=kn_seg, r_ref=R_REF, magnet_order=m,
        options=OPTIONS, min_b1_T=MIN_B1_T,
    )

    extra = [
        {"global_turn": int(plateau_indices[t]),
          "label": str(turn_label[plateau_indices[t]]),
          "I_range_A": float(d["I_range"][plateau_indices[t]]),
          "segment": seg}
        for t in range(len(plateau_indices))
    ]

    rows = build_harmonic_rows(result, C_merged, C_units, ok_main, m, extra)
    df = pd.DataFrame(rows)

    # Group by supercycle
    df["sc_idx"] = -1
    settled_idx = []

    for gi, (gs, ge) in enumerate(d["inj_groups"]):
        group_globals = set(range(gs, ge + 1))
        gmask = df["global_turn"].isin(group_globals) & (df["label"] == "injection")
        df.loc[gmask, "sc_idx"] = gi
        group_rows = df.index[gmask]
        if N_LAST_TURNS_INJ is not None and len(group_rows) > N_LAST_TURNS_INJ:
            settled_idx.extend(group_rows[-N_LAST_TURNS_INJ:])
        else:
            settled_idx.extend(group_rows)

    for gi, (gs, ge) in enumerate(d["fh_groups"]):
        group_globals = set(range(gs, ge + 1))
        gmask = df["global_turn"].isin(group_globals) & (df["label"] == "flat-high")
        df.loc[gmask, "sc_idx"] = gi
        group_rows = df.index[gmask]
        if N_LAST_TURNS_HIGH is not None and len(group_rows) > N_LAST_TURNS_HIGH:
            settled_idx.extend(group_rows[-N_LAST_TURNS_HIGH:])
        else:
            settled_idx.extend(group_rows)

    df_settled = df.loc[sorted(settled_idx)].copy()

    n_before = len(df_settled)
    df_settled, clip_info = mad_sigma_clip(df_settled, "B1_T", N_SIGMA_CLIP, label_col="label")
    n_clipped = n_before - len(df_settled)
    if n_clipped > 0:
        print(f"  Sigma clip ({N_SIGMA_CLIP} MAD sigma): removed {n_clipped} turns ({clip_info})")

    df["TF_TperkA"] = df["B1_T"] / (df["I_mean_A"] / 1000.0)
    df_settled["TF_TperkA"] = df_settled["B1_T"] / (df_settled["I_mean_A"] / 1000.0)

    results[seg] = {"df": df, "df_settled": df_settled}

    print(f"  {seg}: {len(df)} all plateau, {len(df_settled)} settled")
    for lab in ["injection", "flat-high"]:
        n_all = len(df[df["label"] == lab])
        n_set = len(df_settled[df_settled["label"] == lab])
        print(f"    {lab:12s}: {n_all} -> {n_set}")

---
## 9. All-Turn Harmonics vs Time

Process all turns (including ramps) to show B1, b2, b3 evolution.

In [ ]:
all_turn_dfs = {}

for seg in SEGMENTS:
    d = data[seg]
    kn_seg = KN[seg]

    result_all, C_merged_all, C_units_all, ok_main_all = process_kn_pipeline(
        flux_abs_turns=d["flux_abs"], flux_cmp_turns=d["flux_cmp"],
        t_turns=d["t_all"], I_turns=d["I_all"],
        kn=kn_seg, r_ref=R_REF, magnet_order=m,
        options=OPTIONS, min_b1_T=MIN_B1_T,
    )

    extra_all = [{"global_turn": int(i), "segment": seg} for i in range(d["n_turns"])]
    rows_all = build_harmonic_rows(result_all, C_merged_all, C_units_all, ok_main_all, m, extra_all)
    df_all = pd.DataFrame(rows_all)
    df_all["t_mean_s"] = d["t_mean"]
    all_turn_dfs[seg] = df_all
    print(f"{seg}: {d['n_turns']} all-turns processed, ok_main={ok_main_all.sum()}")

# Plot B1, b2, b3 vs time
fig, axes = plt.subplots(3, len(SEGMENTS), figsize=(8 * len(SEGMENTS), 12))
if len(SEGMENTS) == 1:
    axes = axes[:, np.newaxis]

for j, seg in enumerate(SEGMENTS):
    df_all = all_turn_dfs[seg]
    ok = df_all["ok_main"]
    for ax_idx, (col, ylabel) in enumerate([("B1_T", "B1 (T)"), ("b2_units", "b2 (units)"), ("b3_units", "b3 (units)")]):
        ax = axes[ax_idx, j]
        ax.scatter(df_all.loc[ok, "t_mean_s"], df_all.loc[ok, col],
                   s=4, alpha=0.3, color="steelblue", zorder=1)
        ax.set_ylabel(ylabel)
        if ax_idx == 0:
            ax.set_title(f"All-turn evolution -- {seg}")
        if ax_idx == 2:
            ax.set_xlabel("Time (s)")

fig.suptitle("All-Turn Harmonics vs Time", fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

---
## 10. FFMM Golden Standard Validation

Compare pipeline output against FFMM per-turn and average results.

In [ ]:
OPTIONS_FFMM = ('dri', 'rot')
FFMM_ROTATE_EXCLUDES_LAST = True

print("=" * 70)
print("FFMM GOLDEN STANDARD COMPARISON")
print(f"FFMM pipeline options: {OPTIONS_FFMM}")
print(f"legacy_rotate_excludes_last = {FFMM_ROTATE_EXCLUDES_LAST}")
print("=" * 70)

# This section expects FFMM result files alongside the raw data.
# Adjust paths as needed for your measurement.
print("(FFMM validation section -- configure paths for your measurement)")

---
## 11. Main Field (B1)

In [ ]:
fig, axes = plt.subplots(2, len(SEGMENTS), figsize=(8 * len(SEGMENTS), 10))
if len(SEGMENTS) == 1:
    axes = axes[:, np.newaxis]

for j, seg in enumerate(SEGMENTS):
    df = results[seg]["df"]
    df_settled = results[seg]["df_settled"]
    ok = df["ok_main"]
    _scfg = next(sc for sc in SEGMENT_CONFIGS if sc["name"] == seg)
    fringe = " [fringe]" if _scfg["is_fringe"] else ""

    ax = axes[0, j]
    ax.scatter(df.loc[ok, "I_mean_A"], df.loc[ok, "B1_T"], s=8, alpha=0.5, color="steelblue")
    
    ax.set_xlabel("I (A)"); ax.set_ylabel("B1 (T)")
    ax.set_title(f"B1 vs current -- {seg}{fringe}")

    ax = axes[1, j]
    for lab, col, marker in [("injection", "tab:green", "o"), ("flat-high", "tab:blue", "s")]:
        sub = df_settled[(df_settled["label"] == lab) & df_settled["ok_main"]]
        if len(sub) > 0:
            sc_avg = sub.groupby("sc_idx")["B1_T"].agg(["mean", "std"]).reset_index()
            ax.errorbar(sc_avg["sc_idx"], sc_avg["mean"], yerr=sc_avg["std"],
                        fmt=f"{marker}-", markersize=4, capsize=2, color=col, label=lab)
    
    ax.set_xlabel("Supercycle index"); ax.set_ylabel("B1 (T)")
    ax.set_title(f"B1 per supercycle (settled) -- {seg}{fringe}"); ax.legend(fontsize=9)

fig.suptitle("Main Field (B1)", fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

# Statistics
print("\nB1 per operating point (settled turns):")
for seg in SEGMENTS:
    df_settled = results[seg]["df_settled"]
    for lab in ["injection", "flat-high"]:
        sub = df_settled[(df_settled["label"] == lab) & df_settled["ok_main"]]
        if len(sub) > 0:
            print(f"  {seg} {lab:12s}: N={len(sub):4d}, mean={sub['B1_T'].mean():+.6f}, std={sub['B1_T'].std():.6f}")

---
## 12. b2 (Quadrupole) -- first allowed harmonic error

In [ ]:
fig, axes = plt.subplots(2, len(SEGMENTS), figsize=(8 * len(SEGMENTS), 10))
if len(SEGMENTS) == 1:
    axes = axes[:, np.newaxis]

for j, seg in enumerate(SEGMENTS):
    df = results[seg]["df"]
    df_settled = results[seg]["df_settled"]
    ok = df["ok_main"]
    _scfg = next(sc for sc in SEGMENT_CONFIGS if sc["name"] == seg)
    fringe = " [fringe]" if _scfg["is_fringe"] else ""

    ax = axes[0, j]
    ax.scatter(df.loc[ok, "I_mean_A"], df.loc[ok, "b2_units"], s=8, alpha=0.5, color="steelblue")
    ax.axhline(0, color="grey", linewidth=0.5)
    ax.set_xlabel("I (A)"); ax.set_ylabel("b2 (units)")
    ax.set_title(f"b2 vs current -- {seg}{fringe}")

    ax = axes[1, j]
    for lab, col, marker in [("injection", "tab:green", "o"), ("flat-high", "tab:blue", "s")]:
        sub = df_settled[(df_settled["label"] == lab) & df_settled["ok_main"]]
        if len(sub) > 0:
            sc_avg = sub.groupby("sc_idx")["b2_units"].agg(["mean", "std"]).reset_index()
            ax.errorbar(sc_avg["sc_idx"], sc_avg["mean"], yerr=sc_avg["std"],
                        fmt=f"{marker}-", markersize=4, capsize=2, color=col, label=lab)
    ax.axhline(0, color="grey", linewidth=0.5)
    ax.set_xlabel("Supercycle index"); ax.set_ylabel("b2 (units)")
    ax.set_title(f"b2 per supercycle (settled) -- {seg}{fringe}"); ax.legend(fontsize=9)

fig.suptitle("b2 (Quadrupole) -- first allowed harmonic error", fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

# Statistics
print("\nb2 per operating point (settled turns):")
for seg in SEGMENTS:
    df_settled = results[seg]["df_settled"]
    for lab in ["injection", "flat-high"]:
        sub = df_settled[(df_settled["label"] == lab) & df_settled["ok_main"]]
        if len(sub) > 0:
            print(f"  {seg} {lab:12s}: N={len(sub):4d}, mean={sub['b2_units'].mean():+.6f}, std={sub['b2_units'].std():.6f}")

---
## 13. b3 (Sextupole) -- first non-allowed harmonic

In [ ]:
fig, axes = plt.subplots(2, len(SEGMENTS), figsize=(8 * len(SEGMENTS), 10))
if len(SEGMENTS) == 1:
    axes = axes[:, np.newaxis]

for j, seg in enumerate(SEGMENTS):
    df = results[seg]["df"]
    df_settled = results[seg]["df_settled"]
    ok = df["ok_main"]
    _scfg = next(sc for sc in SEGMENT_CONFIGS if sc["name"] == seg)
    fringe = " [fringe]" if _scfg["is_fringe"] else ""

    ax = axes[0, j]
    ax.scatter(df.loc[ok, "I_mean_A"], df.loc[ok, "b3_units"], s=8, alpha=0.5, color="steelblue")
    ax.axhline(0, color="grey", linewidth=0.5)
    ax.set_xlabel("I (A)"); ax.set_ylabel("b3 (units)")
    ax.set_title(f"b3 vs current -- {seg}{fringe}")

    ax = axes[1, j]
    for lab, col, marker in [("injection", "tab:green", "o"), ("flat-high", "tab:blue", "s")]:
        sub = df_settled[(df_settled["label"] == lab) & df_settled["ok_main"]]
        if len(sub) > 0:
            sc_avg = sub.groupby("sc_idx")["b3_units"].agg(["mean", "std"]).reset_index()
            ax.errorbar(sc_avg["sc_idx"], sc_avg["mean"], yerr=sc_avg["std"],
                        fmt=f"{marker}-", markersize=4, capsize=2, color=col, label=lab)
    ax.axhline(0, color="grey", linewidth=0.5)
    ax.set_xlabel("Supercycle index"); ax.set_ylabel("b3 (units)")
    ax.set_title(f"b3 per supercycle (settled) -- {seg}{fringe}"); ax.legend(fontsize=9)

fig.suptitle("b3 (Sextupole) -- first non-allowed harmonic", fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

# Statistics
print("\nb3 per operating point (settled turns):")
for seg in SEGMENTS:
    df_settled = results[seg]["df_settled"]
    for lab in ["injection", "flat-high"]:
        sub = df_settled[(df_settled["label"] == lab) & df_settled["ok_main"]]
        if len(sub) > 0:
            print(f"  {seg} {lab:12s}: N={len(sub):4d}, mean={sub['b3_units'].mean():+.6f}, std={sub['b3_units'].std():.6f}")

---
## 14. Higher Harmonics Overview

Statistics for all harmonics at key operating points (NCS).

In [ ]:
seg = "NCS"
df_settled = results[seg]["df_settled"]
ok = df_settled["ok_main"]

for lab in ["injection", "flat-high"]:
    sub = df_settled[(df_settled["label"] == lab) & ok]
    if len(sub) == 0:
        continue
    print(f"\n=== {lab.upper()} (N={len(sub)} settled turns, {seg}) ===")
    print(f"  {'n':>3s} {'bn mean':>10s} {'bn std':>10s} {'an mean':>10s} {'an std':>10s}")
    print("  " + "-" * 50)
    for nn in range(2, H + 1):
        bn_col = f"b{nn}_units"
        an_col = f"a{nn}_units"
        if bn_col in sub.columns:
            bn_m, bn_s = sub[bn_col].mean(), sub[bn_col].std()
            an_m, an_s = sub[an_col].mean(), sub[an_col].std()
            flag = " *" if abs(bn_m) > 2 * bn_s and abs(bn_m) > 0.5 else ""
            print(f"  {nn:3d} {bn_m:+10.4f} {bn_s:10.4f} {an_m:+10.4f} {an_s:10.4f}{flag}")
    print("  (* = |mean| > 2*std and |mean| > 0.5 units)")

---
## 15. Multipole Spectrum

Bar charts of normal (bn) and skew (an) harmonics.

In [ ]:
seg = "NCS"
df_settled = results[seg]["df_settled"]
ok = df_settled["ok_main"]

operating_points = {}
for lab in ["injection", "flat-high"]:
    sub = df_settled[(df_settled["label"] == lab) & ok]
    if len(sub) > 0:
        operating_points[lab] = sub

n_ops = len(operating_points)
if n_ops > 0:
    fig, axes = plt.subplots(n_ops, 2, figsize=(16, 5 * n_ops))
    if n_ops == 1:
        axes = axes[np.newaxis, :]

    for i, (lab, sub) in enumerate(operating_points.items()):
        orders = list(range(2, H + 1))
        bn_means = [sub[f"b{nn}_units"].mean() for nn in orders]
        an_means = [sub[f"a{nn}_units"].mean() for nn in orders]
        x = np.arange(len(orders))
        w = 0.35

        ax = axes[i, 0]
        ax.bar(x - w/2, bn_means, w, label="bn", color="steelblue", alpha=0.8)
        ax.bar(x + w/2, an_means, w, label="an", color="tab:orange", alpha=0.8)
        ax.axhline(0, color="grey", linewidth=0.5)
        ax.set_xticks(x); ax.set_xticklabels(orders)
        ax.set_xlabel("n"); ax.set_ylabel("Units")
        ax.set_title(f"Spectrum -- {lab} (linear)"); ax.legend(fontsize=8)

        ax = axes[i, 1]
        ax.bar(x - w/2, np.abs(bn_means), w, label="|bn|", color="steelblue", alpha=0.8)
        ax.bar(x + w/2, np.abs(an_means), w, label="|an|", color="tab:orange", alpha=0.8)
        ax.set_yscale("log")
        ax.set_xticks(x); ax.set_xticklabels(orders)
        ax.set_xlabel("n"); ax.set_ylabel("|Units|")
        ax.set_title(f"Spectrum -- {lab} (log)"); ax.legend(fontsize=8)

    fig.suptitle("Multipole Spectrum", fontsize=14, y=1.02)
    plt.tight_layout(); plt.show()
else:
    print("No operating points with data for spectrum plot.")

---
## 16. Transfer Function B1/I

TF = B1 / I (units: T/kA).

In [ ]:
fig, axes = plt.subplots(2, len(SEGMENTS), figsize=(8 * len(SEGMENTS), 10))
if len(SEGMENTS) == 1:
    axes = axes[:, np.newaxis]

tf_summary = {}
for j, seg in enumerate(SEGMENTS):
    df_settled = results[seg]["df_settled"]
    ok = df_settled["ok_main"]
    _scfg = next(sc for sc in SEGMENT_CONFIGS if sc["name"] == seg)
    fringe = " [fringe]" if _scfg["is_fringe"] else ""

    ax = axes[0, j]
    sub_ok = df_settled[ok]
    ax.scatter(sub_ok["I_mean_A"], sub_ok["TF_TperkA"], s=8, alpha=0.5, color="steelblue")
    ax.set_xlabel("I (A)"); ax.set_ylabel("TF = B1/I (T/kA)")
    ax.set_title(f"TF vs current -- {seg}{fringe}")

    ax = axes[1, j]
    ds_tf = {}
    for lab, col, marker in [("injection", "tab:green", "o"), ("flat-high", "tab:blue", "s")]:
        sub = df_settled[(df_settled["label"] == lab) & ok]
        if len(sub) > 0:
            sc_avg = sub.groupby("sc_idx")["TF_TperkA"].agg(["mean", "std"]).reset_index()
            ax.errorbar(sc_avg["sc_idx"], sc_avg["mean"], yerr=sc_avg["std"],
                        fmt=f"{marker}-", markersize=4, capsize=2, color=col, label=lab)
            ds_tf[lab] = sc_avg
    ax.set_xlabel("Supercycle index"); ax.set_ylabel("TF (T/kA)")
    ax.set_title(f"TF per supercycle -- {seg}{fringe}"); ax.legend(fontsize=9)
    tf_summary[seg] = ds_tf

fig.suptitle("Transfer Function B1/I", fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

---
## 17. Apparent vs Differential Inductance

**L_app** = B1/I, **L_d** = dB1/dI (from paired current levels).

In [ ]:
ld_results = {}
for seg in SEGMENTS:
    df_settled = results[seg]["df_settled"]
    ok = df_settled["ok_main"]
    df_inj = df_settled[(df_settled["label"] == "injection") & ok]
    df_fh = df_settled[(df_settled["label"] == "flat-high") & ok]

    if len(df_inj) == 0 or len(df_fh) == 0:
        ld_results[seg] = pd.DataFrame()
        continue

    inj_avg = df_inj.groupby("sc_idx").agg(
        B1_inj=("B1_T", "mean"), I_inj=("I_mean_A", "mean")).reset_index()
    fh_avg = df_fh.groupby("sc_idx").agg(
        B1_fh=("B1_T", "mean"), I_fh=("I_mean_A", "mean")).reset_index()

    merged = inj_avg.merge(fh_avg, on="sc_idx", how="inner")
    if len(merged) == 0:
        ld_results[seg] = pd.DataFrame()
        continue

    merged["Ld_TperkA"] = (merged["B1_fh"] - merged["B1_inj"]) / ((merged["I_fh"] - merged["I_inj"]) / 1000.0)
    ld_results[seg] = merged
    print(f"{seg}: {len(merged)} SC pairs, "
          f"Ld = {merged['Ld_TperkA'].mean():.4f} +/- {merged['Ld_TperkA'].std():.4f} T/kA")

print("\nSaturation check (Ld < L_app(FT) => saturated):")
for seg in SEGMENTS:
    m_df = ld_results.get(seg, pd.DataFrame())
    if len(m_df) == 0:
        continue
    Ld_mean = m_df["Ld_TperkA"].mean()
    fh = results[seg]["df_settled"]
    fh_ok = fh[(fh["label"] == "flat-high") & fh["ok_main"]]
    if len(fh_ok) > 0:
        Lapp_fh = fh_ok["TF_TperkA"].mean()
        ratio = Ld_mean / Lapp_fh
        verdict = "SATURATED" if ratio < 0.99 else "LINEAR"
        print(f"  {seg}: Ld={Ld_mean:.4f}, L_app(FT)={Lapp_fh:.4f}, ratio={ratio:.4f} -> {verdict}")

---
## 18. Eddy Current Settling Analysis

Turn-by-turn B1 for all runs. Eddy currents cause exponential decay.

In [ ]:
# Build per-supercycle injection data
eddy_data = {}

for seg in SEGMENTS:
    d = data[seg]
    df = results[seg]["df"]
    inj = df[df["label"] == "injection"].copy()
    if len(inj) == 0:
        eddy_data[seg] = pd.DataFrame()
        continue

    inj["t_mean_s"] = d["t_mean"][inj["global_turn"].values]
    for sc_id in inj["sc_idx"].unique():
        if sc_id < 0: continue
        mask = inj["sc_idx"] == sc_id
        t0 = inj.loc[mask, "t_mean_s"].min()
        inj.loc[mask, "t_since_inj_start"] = inj.loc[mask, "t_mean_s"] - t0

    for sc_id in inj["sc_idx"].unique():
        if sc_id < 0: continue
        mask = inj["sc_idx"] == sc_id
        inj.loc[mask, "turn_in_group"] = np.arange(mask.sum())

    eddy_data[seg] = inj
    print(f"{seg}: {len(inj)} injection turns across {inj['sc_idx'].nunique()} supercycles")

### Settling curves -- Mean ± STD across supercycles

In [ ]:
_eddy_cols = [("B1_T", "B1 (T)"), ("b2_units", "b2 (units)"), ("b3_units", "b3 (units)")]
fig, axes = plt.subplots(len(SEGMENTS), len(_eddy_cols),
                         figsize=(5 * len(_eddy_cols), 5 * len(SEGMENTS)))
if len(SEGMENTS) == 1:
    axes = axes[np.newaxis, :]

for i, seg in enumerate(SEGMENTS):
    inj = eddy_data[seg]
    for col_idx, (col, ylabel) in enumerate(_eddy_cols):
        ax = axes[i, col_idx]
        if len(inj) == 0:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
            continue
        sc_ids = sorted([s for s in inj["sc_idx"].unique() if s >= 0])
        # Align supercycles by turn index and compute mean/std
        all_turns = {}
        for sc_id in sc_ids:
            sub = inj[inj["sc_idx"] == sc_id].sort_values("turn_in_group")
            for _, row in sub.iterrows():
                tidx = int(row["turn_in_group"])
                all_turns.setdefault(tidx, {"t": [], "y": []})
                all_turns[tidx]["t"].append(row["t_since_inj_start"])
                all_turns[tidx]["y"].append(row[col])
        turn_idxs = sorted(all_turns.keys())
        t_mean = np.array([np.mean(all_turns[k]["t"]) for k in turn_idxs])
        y_mean = np.array([np.mean(all_turns[k]["y"]) for k in turn_idxs])
        y_std = np.array([np.std(all_turns[k]["y"]) for k in turn_idxs])
        # Plot individual SCs faintly
        cmap = plt.cm.tab20(np.linspace(0, 1, max(len(sc_ids), 1)))
        for k, sc_id in enumerate(sc_ids):
            sub = inj[inj["sc_idx"] == sc_id]
            ax.plot(sub["t_since_inj_start"], sub[col], ".",
                    markersize=2, alpha=0.15, color="gray")
        # Plot mean + STD band
        ax.fill_between(t_mean, y_mean - y_std, y_mean + y_std,
                         alpha=0.3, color="tab:blue", label="± 1 STD")
        ax.plot(t_mean, y_mean, "-", linewidth=1.5, color="tab:blue", label="mean")
        ax.set_xlabel("t - t_inj_start (s)"); ax.set_ylabel(ylabel)
        ax.set_title(f"{ylabel.split()[0]} settling -- {SEG_DISPLAY[seg]}")
        ax.legend(fontsize=7)

fig.suptitle("Injection Settling -- Mean ± STD", fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

---
## 19. Multi-Exponential Fits (B1, b2, b3)

Fit 1-, 2-, and 3-exponential eddy models per supercycle for B1, b2, and b3.
Only plateau turns (stable current) are used -- ramp artifacts are excluded.

In [ ]:
from rotating_coil_analyzer.analysis.utility_functions import (
    eddy_model, double_eddy_model, triple_eddy_model,
    validate_eddy_model_selection,
)

def _compute_r2(y, y_pred):
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    return 1 - ss_res / ss_tot if ss_tot > 0 else 0.0

def _aic(n_pts, k, ss_res):
    if n_pts <= k + 1 or ss_res <= 0:
        return np.inf
    aic = n_pts * np.log(ss_res / n_pts) + 2 * k
    return aic + (2 * k * (k + 1)) / max(n_pts - k - 1, 1)

def _fit_123(t, y):
    """Fit 1/2/3-tau models. Returns dict {1: ..., 2: ..., 3: ...}."""
    fr = {}
    n = len(t)
    if n < MIN_INJECTION_TURNS:
        return None
    y_inf = y[-max(5, n // 4):].mean()
    A0 = y[0] - y_inf
    tau0 = max(t[-1] / 3, 0.5)

    # 1-tau
    try:
        p1, _ = curve_fit(eddy_model, t, y, p0=[y_inf, A0, tau0],
                          bounds=([-np.inf, -np.inf, 0.05], [np.inf, np.inf, 1000]), maxfev=10000)
        pred = eddy_model(t, *p1)
        r2 = _compute_r2(y, pred); ss = np.sum((y - pred)**2)
        fr[1] = {"popt": p1, "r2": r2, "aic": _aic(n, 3, ss)}
    except (RuntimeError, ValueError):
        fr[1] = {"popt": None, "r2": 0, "aic": np.inf}

    # 2-tau
    if n >= 10 and fr[1].get("popt") is not None:
        try:
            Bi, Ai, ti = fr[1]["popt"]
            p2, _ = curve_fit(double_eddy_model, t, y,
                              p0=[Bi, Ai*0.5, ti/3, Ai*0.5, ti*2],
                              bounds=([-np.inf,-np.inf,0.05,-np.inf,0.05],
                                      [np.inf,np.inf,500,np.inf,2000]), maxfev=20000)
            if p2[2] > p2[4]:
                p2 = np.array([p2[0], p2[3], p2[4], p2[1], p2[2]])
            pred = double_eddy_model(t, *p2)
            r2 = _compute_r2(y, pred); ss = np.sum((y - pred)**2)
            fr[2] = {"popt": p2, "r2": r2, "aic": _aic(n, 5, ss)}
        except (RuntimeError, ValueError):
            fr[2] = {"popt": None, "r2": 0, "aic": np.inf}
    else:
        fr[2] = {"popt": None, "r2": 0, "aic": np.inf}

    # 3-tau
    if n >= 20 and fr.get(2, {}).get("popt") is not None:
        try:
            B2, A12, t12, A22, t22 = fr[2]["popt"]
            p3, _ = curve_fit(triple_eddy_model, t, y,
                              p0=[B2, A12*0.5, t12/2, A12*0.5, t12*2, A22, t22*2],
                              bounds=([-np.inf,-np.inf,0.05,-np.inf,0.05,-np.inf,0.05],
                                      [np.inf,np.inf,500,np.inf,2000,np.inf,5000]), maxfev=30000)
            taus = [(p3[1],p3[2]), (p3[3],p3[4]), (p3[5],p3[6])]
            taus.sort(key=lambda x: x[1])
            p3 = np.array([p3[0], taus[0][0], taus[0][1], taus[1][0], taus[1][1], taus[2][0], taus[2][1]])
            pred = triple_eddy_model(t, *p3)
            r2 = _compute_r2(y, pred)
            if r2 < 0:
                fr[3] = {"popt": None, "r2": 0, "aic": np.inf}
            else:
                ss = np.sum((y - pred)**2)
                fr[3] = {"popt": p3, "r2": r2, "aic": _aic(n, 7, ss)}
        except (RuntimeError, ValueError):
            fr[3] = {"popt": None, "r2": 0, "aic": np.inf}
    else:
        fr[3] = {"popt": None, "r2": 0, "aic": np.inf}
    return fr

def _print_fit_detail(seg_disp, qlabel, fr, best_ntau):
    for ntau in [1, 2, 3]:
        r = fr.get(ntau, {"r2": 0, "popt": None})
        tag = " <-- BEST" if ntau == best_ntau and r["popt"] is not None and r["r2"] > 0 else ""
        if r["popt"] is not None and r["r2"] > 0:
            p = r["popt"]
            if ntau == 1:
                s = f"tau={p[2]:.3f} s, A={p[1]:.6f}, B_inf={p[0]:.6f}"
            elif ntau == 2:
                s = f"tau1={p[2]:.3f} s, tau2={p[4]:.3f} s, A1={p[1]:.6f}, A2={p[3]:.6f}, B_inf={p[0]:.6f}"
            else:
                s = f"tau1={p[2]:.3f} s, tau2={p[4]:.3f} s, tau3={p[6]:.3f} s, B_inf={p[0]:.6f}"
            print(f"  {seg_disp} {qlabel}: {ntau}-tau R2={r['r2']:.4f}  {s}{tag}")
        else:
            print(f"  {seg_disp} {qlabel}: {ntau}-tau FAILED")

def _print_summary_table(title, results_dict, segments, quantities):
    print(f"\n{title}")
    print("=" * 120)
    print(f"{'Segment':<20s} {'Quantity':<14s} {'Model':>6s} {'R2':>8s} "
          f"{'B_inf':>12s} {'A / A1':>12s} {'tau1 (s)':>10s} {'A2':>12s} {'tau2 (s)':>10s} {'tau3 (s)':>10s}")
    print("-" * 120)
    for seg in segments:
        for col, qlabel in quantities:
            res = results_dict[seg][col]
            bn = res.get("best_ntau")
            bp = res.get("best_popt")
            br = res.get("best_r2", 0)
            if bp is None or br <= 0:
                print(f"{SEG_DISPLAY[seg]:<20s} {qlabel:<14s} {'--':>6s} {'--':>8s}")
                continue
            if bn == 1:
                print(f"{SEG_DISPLAY[seg]:<20s} {qlabel:<14s} {'1-tau':>6s} {br:8.4f} "
                      f"{bp[0]:12.6f} {bp[1]:12.6f} {bp[2]:10.3f}")
            elif bn == 2:
                print(f"{SEG_DISPLAY[seg]:<20s} {qlabel:<14s} {'2-tau':>6s} {br:8.4f} "
                      f"{bp[0]:12.6f} {bp[1]:12.6f} {bp[2]:10.3f} {bp[3]:12.6f} {bp[4]:10.3f}")
            elif bn == 3:
                print(f"{SEG_DISPLAY[seg]:<20s} {qlabel:<14s} {'3-tau':>6s} {br:8.4f} "
                      f"{bp[0]:12.6f} {bp[1]:12.6f} {bp[2]:10.3f} {bp[3]:12.6f} {bp[4]:10.3f} {bp[6]:10.3f}")
    print("=" * 120)

# ---- Fit on MEAN across supercycles (all turns, no outlier removal) ----
EDDY_QUANTITIES = [("B1_T", "B1 (T)"), ("b2_units", "b2 (units)"), ("b3_units", "b3 (units)")]

eddy_fit_all = {}
for seg in SEGMENTS:
    eddy_fit_all[seg] = {}
    inj = eddy_data[seg]
    if len(inj) == 0:
        for col, _ in EDDY_QUANTITIES:
            eddy_fit_all[seg][col] = {"best_ntau": None, "best_popt": None, "best_r2": 0,
                                       "fit_results": {}, "mean_t": None, "mean_y": None, "std_y": None}
        continue

    sc_ids = sorted([s for s in inj["sc_idx"].unique() if s >= 0])
    print(f"{SEG_DISPLAY[seg]}: {len(sc_ids)} supercycles")

    for col, qlabel in EDDY_QUANTITIES:
        # Build mean per turn index
        all_turns = {}
        for sc_id in sc_ids:
            sub = inj[inj["sc_idx"] == sc_id].sort_values("turn_in_group")
            for _, row in sub.iterrows():
                tidx = int(row["turn_in_group"])
                all_turns.setdefault(tidx, {"t": [], "y": []})
                all_turns[tidx]["t"].append(row["t_since_inj_start"])
                all_turns[tidx]["y"].append(row[col])
        turn_idxs = sorted(all_turns.keys())
        t_mean = np.array([np.mean(all_turns[k]["t"]) for k in turn_idxs])
        y_mean = np.array([np.mean(all_turns[k]["y"]) for k in turn_idxs])
        y_std = np.array([np.std(all_turns[k]["y"]) for k in turn_idxs])

        # Fit on full mean (no outlier removal)
        best_ntau, best_popt, best_r2 = 1, None, 0.0
        fr = _fit_123(t_mean, y_mean)
        if fr is not None:
            best_ntau, _ = validate_eddy_model_selection(fr)
            if best_ntau in fr and fr[best_ntau].get("popt") is not None:
                best_popt = fr[best_ntau]["popt"]
                best_r2 = fr[best_ntau]["r2"]
            _print_fit_detail(SEG_DISPLAY[seg], qlabel, fr, best_ntau)
        else:
            fr = {}

        eddy_fit_all[seg][col] = {
            "best_ntau": best_ntau, "best_popt": best_popt, "best_r2": best_r2,
            "fit_results": fr, "mean_t": t_mean, "mean_y": y_mean, "std_y": y_std,
        }

_print_summary_table("BEFORE OUTLIER REMOVAL (fit on mean across SCs, all turns)",
                     eddy_fit_all, SEGMENTS, EDDY_QUANTITIES)

### Multi-Tau Fit Comparison Plots

In [ ]:
# Plot 1/2/3-tau fits on the mean for all segments
_models_t = {1: eddy_model, 2: double_eddy_model, 3: triple_eddy_model}
_colors_t = {1: "tab:green", 2: "tab:orange", 3: "tab:red"}
_labels_t = {1: "1-tau", 2: "2-tau", 3: "3-tau"}

for seg in SEGMENTS:
    fig, axes = plt.subplots(1, len(EDDY_QUANTITIES), figsize=(6 * len(EDDY_QUANTITIES), 5))
    if len(EDDY_QUANTITIES) == 1:
        axes = [axes]

    for ax, (col, qlabel) in zip(axes, EDDY_QUANTITIES):
        res = eddy_fit_all[seg][col]
        tm = res.get("mean_t")
        ym = res.get("mean_y")
        ys = res.get("std_y")
        if tm is None or len(tm) == 0:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
            continue

        # Mean ± STD
        ax.fill_between(tm, ym - ys, ym + ys, alpha=0.2, color="tab:blue")
        ax.plot(tm, ym, ".", markersize=4, color="tab:blue", alpha=0.6, label="mean")

        # Overlay all converged fits
        fr = res.get("fit_results", {})
        if fr:
            t_fine = np.linspace(tm.min(), tm.max(), 300)
            for ntau in [1, 2, 3]:
                r = fr.get(ntau, {"popt": None, "r2": 0})
                if r["popt"] is not None and r["r2"] > 0.01:
                    is_best = res["best_ntau"] == ntau
                    ax.plot(t_fine, _models_t[ntau](t_fine, *r["popt"]),
                            color=_colors_t[ntau], linewidth=2 if is_best else 1,
                            linestyle="-" if is_best else "--",
                            label=f"{_labels_t[ntau]} R\u00b2={r['r2']:.4f}" + (" *" if is_best else ""))
        ax.legend(fontsize=7); ax.set_xlabel("t (s)"); ax.set_ylabel(qlabel)
        ax.set_title(f"{qlabel.split()[0]} -- {SEG_DISPLAY[seg]}")

    fig.suptitle(f"Multi-tau fit comparison (mean, before outlier removal) -- {SEG_DISPLAY[seg]}", fontsize=12, y=1.02)
    plt.tight_layout(); plt.show()

---
## 20. Outlier-Cleaned Fits (Mean ± STD)

Remove ramp artifacts using residual-based clipping:
1. Compute mean across supercycles per turn index
2. Fit a preliminary 1-tau model to the mean
3. Remove turns whose residuals exceed 3.5× MAD of residuals
4. Refit 1/2/3-tau models on the cleaned mean and select best by AICc

In [ ]:
# --- Outlier removal (residual-based) and clean refit ---
from rotating_coil_analyzer.analysis.utility_functions import (
    eddy_model, double_eddy_model, triple_eddy_model,
    validate_eddy_model_selection,
)

def _compute_r2_c(y, y_pred):
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    return 1 - ss_res / ss_tot if ss_tot > 0 else 0.0

def _aic_c(n_pts, k, ss_res):
    if n_pts <= k + 1 or ss_res <= 0:
        return np.inf
    aic = n_pts * np.log(ss_res / n_pts) + 2 * k
    return aic + (2 * k * (k + 1)) / max(n_pts - k - 1, 1)

def _residual_clip(t, y, threshold=3.5):
    """Remove outliers based on residuals from a preliminary 1-tau fit.

    Unlike raw MAD, this preserves the real exponential signal and only
    removes points that deviate from the expected trend (ramp artifacts).
    """
    if len(t) < 5:
        return np.ones(len(t), dtype=bool)
    # Preliminary fit
    try:
        y_inf_est = y[-max(3, len(y) // 4):].mean()
        A_est = y[0] - y_inf_est
        tau_est = max(t[-1] / 3, 0.5)
        popt, _ = curve_fit(eddy_model, t, y, p0=[y_inf_est, A_est, tau_est],
                             bounds=([-np.inf, -np.inf, 0.05], [np.inf, np.inf, 1000]),
                             maxfev=5000)
        residuals = y - eddy_model(t, *popt)
    except (RuntimeError, ValueError):
        # If preliminary fit fails, fall back to simple differences
        residuals = np.diff(y, prepend=y[0])

    med_r = np.median(residuals)
    mad_r = np.median(np.abs(residuals - med_r))
    if mad_r < 1e-15:
        return np.ones(len(t), dtype=bool)
    modified_z = 0.6745 * np.abs(residuals - med_r) / mad_r
    return modified_z < threshold

_models_c = {1: eddy_model, 2: double_eddy_model, 3: triple_eddy_model}
_colors_c = {1: "tab:green", 2: "tab:orange", 3: "tab:red"}
_labels_c = {1: "1-tau", 2: "2-tau", 3: "3-tau"}
EDDY_QUANTITIES_C = [("B1_T", "B1 (T)"), ("b2_units", "b2 (units)"), ("b3_units", "b3 (units)")]

eddy_clean_results = {}
for seg in SEGMENTS:
    eddy_clean_results[seg] = {}
    inj = eddy_data[seg]
    if len(inj) == 0:
        for col, _ in EDDY_QUANTITIES_C:
            eddy_clean_results[seg][col] = {"mean_t": None}
        continue

    sc_ids = sorted([s for s in inj["sc_idx"].unique() if s >= 0])

    for col, qlabel in EDDY_QUANTITIES_C:
        # Build per-turn arrays across SCs (aligned by turn_in_group)
        all_turns = {}
        for sc_id in sc_ids:
            sub = inj[inj["sc_idx"] == sc_id].sort_values("turn_in_group")
            for _, row in sub.iterrows():
                tidx = int(row["turn_in_group"])
                all_turns.setdefault(tidx, {"t": [], "y": []})
                all_turns[tidx]["t"].append(row["t_since_inj_start"])
                all_turns[tidx]["y"].append(row[col])

        turn_idxs = sorted(all_turns.keys())
        t_mean = np.array([np.mean(all_turns[k]["t"]) for k in turn_idxs])
        y_mean = np.array([np.mean(all_turns[k]["y"]) for k in turn_idxs])
        y_std = np.array([np.std(all_turns[k]["y"]) for k in turn_idxs])

        # Residual-based clipping (preserves exponential trend, removes ramp artifacts)
        ok = _residual_clip(t_mean, y_mean, threshold=3.5)
        tc, yc, ysc = t_mean[ok], y_mean[ok], y_std[ok]

        n_removed = int((~ok).sum())
        if n_removed > 0:
            print(f"{SEG_DISPLAY[seg]} {qlabel}: removed {n_removed}/{len(t_mean)} outlier turns")

        # Fit 1/2/3-tau on the cleaned mean
        best_ntau, best_popt, best_r2 = 1, None, 0.0
        fit_results_c = {}
        if len(tc) >= MIN_INJECTION_TURNS:
            y_inf_est = yc[-max(5, len(yc) // 4):].mean()
            A_est = yc[0] - y_inf_est
            tau_est = max(tc[-1] / 3, 0.5)

            # 1-tau
            try:
                popt1, _ = curve_fit(eddy_model, tc, yc, p0=[y_inf_est, A_est, tau_est],
                                     bounds=([-np.inf, -np.inf, 0.05], [np.inf, np.inf, 1000]),
                                     maxfev=10000)
                pred = eddy_model(tc, *popt1)
                r2 = _compute_r2_c(yc, pred)
                ss = np.sum((yc - pred) ** 2)
                fit_results_c[1] = {"popt": popt1, "r2": r2, "aic": _aic_c(len(tc), 3, ss)}
            except (RuntimeError, ValueError):
                fit_results_c[1] = {"popt": None, "r2": 0, "aic": np.inf}

            # 2-tau
            if len(tc) >= 10 and fit_results_c[1].get("popt") is not None:
                try:
                    Bi, Ai, ti = fit_results_c[1]["popt"]
                    popt2, _ = curve_fit(double_eddy_model, tc, yc,
                                         p0=[Bi, Ai * 0.5, ti / 3, Ai * 0.5, ti * 2],
                                         bounds=([-np.inf, -np.inf, 0.05, -np.inf, 0.05],
                                                 [np.inf, np.inf, 500, np.inf, 2000]),
                                         maxfev=20000)
                    if popt2[2] > popt2[4]:
                        popt2 = np.array([popt2[0], popt2[3], popt2[4], popt2[1], popt2[2]])
                    pred = double_eddy_model(tc, *popt2)
                    r2 = _compute_r2_c(yc, pred)
                    ss = np.sum((yc - pred) ** 2)
                    fit_results_c[2] = {"popt": popt2, "r2": r2, "aic": _aic_c(len(tc), 5, ss)}
                except (RuntimeError, ValueError):
                    fit_results_c[2] = {"popt": None, "r2": 0, "aic": np.inf}
            else:
                fit_results_c[2] = {"popt": None, "r2": 0, "aic": np.inf}

            # 3-tau
            if len(tc) >= 20 and fit_results_c.get(2, {}).get("popt") is not None:
                try:
                    B2, A12, t12, A22, t22 = fit_results_c[2]["popt"]
                    popt3, _ = curve_fit(triple_eddy_model, tc, yc,
                                         p0=[B2, A12*0.5, t12/2, A12*0.5, t12*2, A22, t22*2],
                                         bounds=([-np.inf, -np.inf, 0.05, -np.inf, 0.05, -np.inf, 0.05],
                                                 [np.inf, np.inf, 500, np.inf, 2000, np.inf, 5000]),
                                         maxfev=30000)
                    taus = [(popt3[1], popt3[2]), (popt3[3], popt3[4]), (popt3[5], popt3[6])]
                    taus.sort(key=lambda x: x[1])
                    popt3 = np.array([popt3[0], taus[0][0], taus[0][1],
                                      taus[1][0], taus[1][1], taus[2][0], taus[2][1]])
                    pred = triple_eddy_model(tc, *popt3)
                    r2 = _compute_r2_c(yc, pred)
                    if r2 < 0:  # Divergent fit
                        fit_results_c[3] = {"popt": None, "r2": 0, "aic": np.inf}
                    else:
                        ss = np.sum((yc - pred) ** 2)
                        fit_results_c[3] = {"popt": popt3, "r2": r2, "aic": _aic_c(len(tc), 7, ss)}
                except (RuntimeError, ValueError):
                    fit_results_c[3] = {"popt": None, "r2": 0, "aic": np.inf}
            else:
                fit_results_c[3] = {"popt": None, "r2": 0, "aic": np.inf}

            # Select best model with overfitting guard
            best_ntau, selection_reason = validate_eddy_model_selection(fit_results_c)
            if best_ntau in fit_results_c and fit_results_c[best_ntau].get("popt") is not None:
                best_popt = fit_results_c[best_ntau]["popt"]
                best_r2 = fit_results_c[best_ntau]["r2"]

            reason_tag = f"  [{selection_reason}]" if selection_reason != "OK" else ""
            for ntau in [1, 2, 3]:
                r = fit_results_c.get(ntau, {"r2": 0, "popt": None})
                tag = f" <-- BEST{reason_tag}" if ntau == best_ntau and r["popt"] is not None and r["r2"] > 0 else ""
                if r["popt"] is not None and r["r2"] > 0:
                    popt = r["popt"]
                    if ntau == 1:
                        tau_str = f"tau={popt[2]:.3f} s, A={popt[1]:.6f}, B_inf={popt[0]:.6f}"
                    elif ntau == 2:
                        tau_str = (f"tau1={popt[2]:.3f} s, tau2={popt[4]:.3f} s, "
                                   f"A1={popt[1]:.6f}, A2={popt[3]:.6f}, B_inf={popt[0]:.6f}")
                    else:
                        tau_str = (f"tau1={popt[2]:.3f} s, tau2={popt[4]:.3f} s, tau3={popt[6]:.3f} s, "
                                   f"B_inf={popt[0]:.6f}")
                    print(f"  {SEG_DISPLAY[seg]} {qlabel}: {ntau}-tau R2={r['r2']:.4f}  {tau_str}{tag}")
                else:
                    print(f"  {SEG_DISPLAY[seg]} {qlabel}: {ntau}-tau FAILED")

        eddy_clean_results[seg][col] = {
            "mean_t": tc, "mean_y": yc, "std_y": ysc,
            "full_t": t_mean, "full_y": y_mean, "full_std": y_std,
            "best_ntau": best_ntau, "best_popt": best_popt, "best_r2": best_r2,
            "fit_results": fit_results_c,
        }

# --- Summary table with tau values ---
print("\n" + "=" * 120)
print(f"{'Segment':<20s} {'Quantity':<14s} {'Model':>6s} {'R2':>8s} "
      f"{'B_inf':>12s} {'A / A1':>12s} {'tau1 (s)':>10s} {'A2':>12s} {'tau2 (s)':>10s} {'tau3 (s)':>10s}")
print("-" * 120)
for seg in SEGMENTS:
    for col, qlabel in EDDY_QUANTITIES_C:
        res = eddy_clean_results[seg][col]
        bn = res.get("best_ntau")
        bp = res.get("best_popt")
        br = res.get("best_r2", 0)
        if bp is None or br <= 0:
            print(f"{SEG_DISPLAY[seg]:<20s} {qlabel:<14s} {'--':>6s} {'--':>8s}")
            continue
        if bn == 1:
            print(f"{SEG_DISPLAY[seg]:<20s} {qlabel:<14s} {'1-tau':>6s} {br:8.4f} "
                  f"{bp[0]:12.6f} {bp[1]:12.6f} {bp[2]:10.3f}")
        elif bn == 2:
            print(f"{SEG_DISPLAY[seg]:<20s} {qlabel:<14s} {'2-tau':>6s} {br:8.4f} "
                  f"{bp[0]:12.6f} {bp[1]:12.6f} {bp[2]:10.3f} {bp[3]:12.6f} {bp[4]:10.3f}")
        elif bn == 3:
            print(f"{SEG_DISPLAY[seg]:<20s} {qlabel:<14s} {'3-tau':>6s} {br:8.4f} "
                  f"{bp[0]:12.6f} {bp[1]:12.6f} {bp[2]:10.3f} {bp[3]:12.6f} {bp[4]:10.3f} {bp[6]:10.3f}")
print("=" * 120)
for seg in SEGMENTS:
    fig, axes = plt.subplots(1, len(EDDY_QUANTITIES_C), figsize=(6 * len(EDDY_QUANTITIES_C), 5))
    if len(EDDY_QUANTITIES_C) == 1:
        axes = [axes]

    for ax, (col, qlabel) in zip(axes, EDDY_QUANTITIES_C):
        res = eddy_clean_results[seg][col]
        if res.get("mean_t") is None or len(res.get("mean_t", [])) == 0:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
            continue

        ft, fy, fs = res["full_t"], res["full_y"], res["full_std"]
        tc, yc, ysc = res["mean_t"], res["mean_y"], res["std_y"]

        # Full data (gray, with outliers)
        ax.fill_between(ft, fy - fs, fy + fs, alpha=0.08, color="gray")
        ax.plot(ft, fy, ".", markersize=3, alpha=0.25, color="gray", label="all turns")

        # Cleaned mean ± STD
        ax.fill_between(tc, yc - ysc, yc + ysc, alpha=0.25, color="tab:blue")
        ax.plot(tc, yc, "o", markersize=3, color="tab:blue", label="cleaned mean")

        # Fit curves
        fr = res.get("fit_results", {})
        if fr:
            t_fine = np.linspace(tc.min(), tc.max(), 300)
            for ntau in [1, 2, 3]:
                r = fr.get(ntau, {"popt": None, "r2": 0})
                if r["popt"] is not None and r["r2"] > 0.01:
                    is_best = res["best_ntau"] == ntau
                    ax.plot(t_fine, _models_c[ntau](t_fine, *r["popt"]),
                            color=_colors_c[ntau], linewidth=2.0 if is_best else 1.0,
                            linestyle="-" if is_best else "--",
                            label=f"{_labels_c[ntau]} R\u00b2={r['r2']:.4f}" + (" *" if is_best else ""))

        ax.set_xlabel("t (s)"); ax.set_ylabel(qlabel)
        ax.set_title(f"{qlabel.split()[0]} -- {SEG_DISPLAY[seg]}")
        ax.legend(fontsize=7)

    fig.suptitle(f"Cleaned Fit: Mean \u00b1 STD -- {SEG_DISPLAY[seg]}", fontsize=13, y=1.02)
    plt.tight_layout(); plt.show()

_print_summary_table("AFTER OUTLIER REMOVAL (fit on mean, ramp artifacts removed)",
                     eddy_clean_results, SEGMENTS, EDDY_QUANTITIES_C)

### Scientific Observations -- Eddy Settling in Fringe vs Main Body

**B1 (dipole field):**
- Both fringe and main-body segments show clear exponential settling after the
  current ramp stops.  The 2-tau model is generally preferred (AICc), indicating
  at least two conducting components with distinct time constants (fast ~1-2 s,
  slow ~3-5 s).
- B1 eddy amplitudes are proportional to the local field level: fringe (~0.3 T)
  has smaller absolute eddy amplitudes than main body (~1.8 T), but the relative
  decay is similar because the eddy mechanism is the same laminated yoke.

**b2 (quadrupole component):**
- b2 shows no significant settling trend in either segment (R² < 0.1 typically).
  This is expected: b2 is a "forbidden" harmonic in a dipole with midplane
  symmetry.  Any residual b2 is dominated by random noise and coil positioning
  imperfections, not eddy currents.

**b3 (sextupole component):**
- In the **fringe field** segment, b3 shows clear exponential settling with
  R² ~ 0.97, amplitude ~1-2 units, tau ~1.6-3.0 s.  This is because the fringe
  has a large DC sextupole (~5 units) arising from the nonlinear field rolloff at
  the magnet end.  Eddy currents modulate this geometric sextupole.
- In the **main body**, b3 is near zero at injection (by design), so the eddy
  perturbation (~0.01 units) is buried in measurement noise (~0.15 units).
  Fits return R² < 0.1 and are physically meaningless.

**Outlier interpretation:**
- The first 1-3 turns after ramp end often deviate from the exponential trend.
  These are NOT random errors -- they capture the fast transient that a 1-tau
  model cannot represent.  The 2-tau fit absorbs most of these "outliers" via its
  fast component.  MAD clipping further removes residual ramp-to-plateau
  transition artefacts, improving fit stability.

**Practical implication:**
- For harmonic measurements, use the main-body segment and wait ≥ 5×tau_slow
  after the ramp to ensure eddies are negligible.  With tau_slow ~ 3-5 s at
  injection, a settling time of 15-25 s (30-50 turns at 2 Hz) is conservative.
- The fringe segment is valuable for studying eddy dynamics (better SNR for b3)
  but its multipole values are NOT representative of the beam region.

---
## 21. Settling Bias Analysis

How b2/b3 averages change with averaging window.

In [ ]:
seg = "NCS"
inj = eddy_data[seg]
if len(inj) > 0 and "turn_in_group" in inj.columns:
    sc_ids = sorted([s for s in inj["sc_idx"].unique() if s >= 0])
    max_turns = inj.groupby("sc_idx").size().min()
    n_last_values = list(range(1, max_turns + 1))

    bias_b3, bias_b2 = [], []
    for n_last in n_last_values:
        b3_m, b2_m = [], []
        for sc_id in sc_ids:
            sub = inj[inj["sc_idx"] == sc_id].sort_values("turn_in_group")
            tail = sub.tail(n_last)
            if len(tail) > 0 and tail["ok_main"].any():
                ok_t = tail[tail["ok_main"]]
                b3_m.append(ok_t["b3_units"].mean())
                b2_m.append(ok_t["b2_units"].mean())
        bias_b3.append(np.mean(b3_m) if b3_m else np.nan)
        bias_b2.append(np.mean(b2_m) if b2_m else np.nan)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(n_last_values, bias_b3, "o-", markersize=4, color="tab:blue")
    axes[0].set_xlabel("N_LAST"); axes[0].set_ylabel("b3 mean (units)")
    axes[0].set_title("b3 bias vs averaging window")
    axes[1].plot(n_last_values, bias_b2, "o-", markersize=4, color="tab:orange")
    axes[1].set_xlabel("N_LAST"); axes[1].set_ylabel("b2 mean (units)")
    axes[1].set_title("b2 bias vs averaging window")
    fig.suptitle("Settling Bias Analysis", fontsize=13, y=1.02)
    plt.tight_layout(); plt.show()
else:
    print("No injection data for bias analysis.")

---
## 22. N_LAST Sensitivity Study

Scan N_LAST and show convergence.

In [ ]:
seg = "NCS"
d = data[seg]
_df = results[seg]["df"]
inj_all = _df[_df["label"] == "injection"].copy()

if len(inj_all) > 0:
    turns_per_sc = inj_all.groupby("sc_idx").size()
    max_n_last = int(turns_per_sc.min())
    n_last_scan = list(range(1, max_n_last + 1))

    scan_results = {"B1_T": [], "b2_units": [], "b3_units": []}
    for n_last in n_last_scan:
        settled_idx = []
        for sc_id in inj_all["sc_idx"].unique():
            if sc_id < 0: continue
            group_rows = inj_all.index[inj_all["sc_idx"] == sc_id]
            if len(group_rows) > n_last:
                settled_idx.extend(group_rows[-n_last:])
            else:
                settled_idx.extend(group_rows)
        sub = inj_all.loc[sorted(settled_idx)]
        sub_ok = sub[sub["ok_main"]]
        for col in ["B1_T", "b2_units", "b3_units"]:
            scan_results[col].append(sub_ok[col].mean() if len(sub_ok) > 0 else np.nan)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, (col, ylabel, color) in zip(axes, [
            ("B1_T", "B1 (T)", "steelblue"),
            ("b2_units", "b2 (units)", "tab:orange"),
            ("b3_units", "b3 (units)", "tab:green")]):
        ax.plot(n_last_scan, scan_results[col], "o-", markersize=3, color=color)
        ax.axvline(N_LAST_TURNS_INJ, color="red", linestyle="--", linewidth=1,
                    label=f"N_LAST={N_LAST_TURNS_INJ}")
        ax.set_xlabel("N_LAST"); ax.set_ylabel(ylabel)
        ax.set_title(f"{ylabel.split()[0]} vs N_LAST"); ax.legend(fontsize=8)
    fig.suptitle("N_LAST Sensitivity", fontsize=13, y=1.02)
    plt.tight_layout(); plt.show()
else:
    print("No injection data for N_LAST sensitivity.")

---
## 23. Comprehensive Statistics Table

In [ ]:
print("=" * 70)
print("SPS MBB Dipole -- 26 GeV MD1 Extended NCS")
print("=" * 70)
print(f"Options: {OPTIONS}")
print(f"cel/fed: {diag.recommendation}")

for seg in SEGMENTS:
    d = data[seg]
    df_settled = results[seg]["df_settled"]
    _scfg = next(sc for sc in SEGMENT_CONFIGS if sc["name"] == seg)
    fringe = " [FRINGE]" if _scfg["is_fringe"] else ""

    print(f"\n--- {seg}{fringe} ---")
    print(f"  Total turns: {d['n_turns']}, Plateau: {d['is_plateau'].sum()}")

    for lab in ["injection", "flat-high"]:
        sub = df_settled[(df_settled["label"] == lab) & df_settled["ok_main"]]
        if len(sub) > 0:
            tf = sub["B1_T"].mean() / (sub["I_mean_A"].mean() / 1e3)
            print(f"  {lab:12s}: N={len(sub):4d}, I={sub['I_mean_A'].mean():.1f} A, "
                  f"B1={sub['B1_T'].mean():+.6f} T, "
                  f"b2={sub['b2_units'].mean():+.3f}, b3={sub['b3_units'].mean():+.3f}, "
                  f"TF={tf:.4f} T/kA")

---
## 24. Analysis Choices Summary

Document all analysis parameters for reproducibility.

In [ ]:
import datetime
print("ANALYSIS CHOICES")
print("=" * 60)
print(f"Generated    : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Title        : SPS MBB Dipole -- 26 GeV MD1 Extended NCS")
print(f"Segments     : {SEGMENTS}")
print(f"Magnet order : {MAGNET_ORDER}")
print(f"R_ref        : {R_REF} m")
print(f"Samples/turn : {SAMPLES_PER_TURN}")
print(f"OPTIONS      : {OPTIONS}")
print(f"MIN_B1_T     : {MIN_B1_T}")
print(f"N_SIGMA_CLIP : {N_SIGMA_CLIP}")

---
## 25. CSV Export

In [ ]:
out_dir = REPO_ROOT / "output" / "MBB/2026-02-06_supercycle/03_26_extended"
out_dir.mkdir(parents=True, exist_ok=True)

for seg in SEGMENTS:
    df_all = results[seg]["df"]
    df_settled = results[seg]["df_settled"]

    fname = f"MBB_{seg}_streaming_plateau.csv"
    df_all.to_csv(out_dir / fname, index=False)
    print(f"Wrote {out_dir / fname}  ({len(df_all)} rows)")

    fname_s = f"MBB_{seg}_streaming_settled.csv"
    df_settled.to_csv(out_dir / fname_s, index=False)
    print(f"Wrote {out_dir / fname_s}  ({len(df_settled)} rows)")

# Export eddy multi-tau fit results (mean-based)
_eddy_rows = []
for seg in SEGMENTS:
    for col, qlabel in EDDY_QUANTITIES:
        res = eddy_fit_all.get(seg, {}).get(col, {})
        bn = res.get("best_ntau")
        bp = res.get("best_popt")
        br = res.get("best_r2", 0)
        if bp is None or br <= 0:
            continue
        row = {"segment": seg, "quantity": col, "model": f"{bn}-tau", "R2": br, "B_inf": bp[0]}
        if bn >= 1:
            row["A1"] = bp[1]; row["tau1_s"] = bp[2]
        if bn >= 2:
            row["A2"] = bp[3]; row["tau2_s"] = bp[4]
        if bn >= 3:
            row["A3"] = bp[5]; row["tau3_s"] = bp[6]
        _eddy_rows.append(row)
if _eddy_rows:
    _df_eddy = pd.DataFrame(_eddy_rows)
    fname_e = "eddy_fits_mean.csv"
    _df_eddy.to_csv(out_dir / fname_e, index=False)
    print(f"Wrote {out_dir / fname_e}  ({len(_df_eddy)} rows)")

    # Export injection raw data for eddy analysis
    inj = eddy_data.get(seg, pd.DataFrame())
    if len(inj) > 0:
        fname_inj = f"eddy_injection_{seg}.csv"
        inj.to_csv(out_dir / fname_inj, index=False)
        print(f"Wrote {out_dir / fname_inj}  ({len(inj)} rows)")

print("\nDone.")